# Session SDK Demo

O novo SDK unifica sandbox + tracing + AI agent numa unica classe.

**Antes** (agent_test.ipynb): ~200 linhas, 6 imports, setup manual, try/finally...

**Agora**: ~10 linhas. Tudo automatico.

In [ ]:
# Setup path + instalar openai (correr uma vez)
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

!pip install openai -q

In [ ]:
# API key
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # <-- coloque sua key aqui

## Exemplo 1 — Agent loop completo (OpenAI)

Uma unica celula faz tudo: cria sandbox, injeta codigo com bugs, corre o agente AI, e finaliza com score.

In [ ]:
from openai import OpenAI
from lunar_sandbox import Session, openai_adapter

client = OpenAI()

with Session("fix-bugs", image="python:3.12-slim") as s:
    # Setup: instalar pytest e injetar ficheiros com bugs
    s.run("pip install -q pytest")

    s.write_file("main.py", '''def divide(a, b):
    """Divide a by b."""
    return a * b  # BUG: deveria ser a / b

def reverse_string(s):
    """Reverse a string."""
    return s  # BUG: deveria inverter
''')

    s.write_file("test_main.py", '''from main import divide, reverse_string

def test_divide_basic():
    assert divide(10, 2) == 5.0

def test_divide_negative():
    assert divide(-6, 3) == -2.0

def test_divide_float():
    assert divide(1, 3) == 1/3

def test_reverse_hello():
    assert reverse_string("hello") == "olleh"

def test_reverse_empty():
    assert reverse_string("") == ""

def test_reverse_single():
    assert reverse_string("a") == "a"
''')

    # Confirmar que testes falham
    print("=== Testes ANTES ===")
    print(s.run("cd /workspace && python -m pytest test_main.py -v"))

    # Agente AI corrige os bugs (tudo auto-traced no dashboard!)
    answer = s.agent_loop(
        task="Fix the bugs in main.py so all tests in test_main.py pass. Run pytest to verify.",
        call_llm=openai_adapter(client, model="gpt-4o-mini"),
    )
    print(f"\nAgent: {answer}")

    # Verificar resultado
    result = s.run("cd /workspace && python -m pytest test_main.py -v")
    print("=== Testes DEPOIS ===")
    print(result)

    passed = "passed" in result and "failed" not in result
    s.finish(outcome="completed" if passed else "failed", score=1.0 if passed else 0.0)

## Exemplo 2 — Uso manual (sem agent loop)

Para quando queres controlar o loop tu mesmo, ou usar sem AI.

In [ ]:
from lunar_sandbox import Session

with Session("manual-demo") as s:
    # Operacoes directas (tudo auto-traced)
    s.run("pip install -q requests")
    s.write_file("hello.py", 'print("Hello from sandbox!")')
    output = s.run("python /workspace/hello.py")
    print(output)

    files = s.list_files()
    print(f"Files: {files}")

    content = s.read_file("hello.py")
    print(f"Content: {content}")

    s.finish(outcome="completed", score=1.0)

## Exemplo 3 — Loop manual com tools (qualquer provider)

Usa `s.tools()` para pegar as tool definitions e `s.call_tool()` para executar.

In [ ]:
import json
from openai import OpenAI
from lunar_sandbox import Session

client = OpenAI()

with Session("custom-loop") as s:
    tools = s.tools(format="openai")  # tool definitions prontas para OpenAI
    messages = [
        {"role": "system", "content": "You are a helpful coding assistant."},
        {"role": "user", "content": "Create a file hello.py that prints 'Hello World', then run it."},
    ]

    for _ in range(10):
        resp = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"Agent: {msg.content}")
            break

        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = s.call_tool(tc.function.name, args)  # executa + trace automatico
            print(f"[{tc.function.name}] -> {result[:200]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    s.finish(score=1.0)